# Waveform Uniformity Model

Predicting nozzle-to-nozzle variability (`sd_std`) from waveform parameters.  
Target: find which parameter combinations produce the most consistent print output across all nozzles.

Follows CRISP-DM phase 4 (Modeling). See `documentation/modeling-approach.md` for design decisions.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_error
from xgboost import XGBRegressor

## 1. Load and aggregate

The processed dataset has one row per printed sheet. Each waveform condition was tested on ~6 replicate sheets.  
We average `sd_std` over those replicates so each row represents the true uniformity of a waveform setting, not a single noisy measurement.

In [ ]:
df = pd.read_parquet('../../data/processed/waveform_tuning_row_summary.parquet')

condition_cols = ['Color$', 'HeadIdx#', 'V', 'F_r', 'dt2', 'Coverage#']
agg = df.groupby(condition_cols)['sd_std'].mean().reset_index()
agg = agg.rename(columns={'sd_std': 'sd_std_mean'})

print(f'Raw rows:        {len(df):,}')
print(f'After aggregation: {len(agg):,} conditions')

## 2. Feature engineering

In [ ]:
# One-hot encode color
color_dummies = pd.get_dummies(agg['Color$'], prefix='color', drop_first=False)

# Base numeric features
base = agg[['V', 'F_r', 'dt2', 'Coverage#']].copy()

# Interaction and polynomial terms motivated by the coverage analysis findings:
# - V and F_r are physically coupled (F_r range shifts with V in the experimental design)
# - dt2 and Coverage interact in their effect on ink deposition
# - V and F_r both show non-linear effects on sd_std
base['V_x_Fr']          = agg['V'] * agg['F_r']
base['dt2_x_coverage']  = agg['dt2'] * agg['Coverage#']
base['V_sq']            = agg['V'] ** 2
base['Fr_sq']           = agg['F_r'] ** 2

X = pd.concat([base, color_dummies], axis=1)
y = agg['sd_std_mean']

print(f'Features: {X.columns.tolist()}')
print(f'Target range: {y.min():.5f} – {y.max():.5f}')

## 3. Train / test split

Split by `HeadIdx#`: train on chips 1–24, test on chips 25–30.  
This tests whether the model generalises to unseen physical printheads — the realistic scenario for deployment.

In [ ]:
train_mask = agg['HeadIdx#'] <= 24

X_train, X_test = X[train_mask], X[~train_mask]
y_train, y_test = y[train_mask], y[~train_mask]

print(f'Train: {len(X_train):,} rows ({train_mask.mean():.0%})')
print(f'Test:  {len(X_test):,} rows ({(~train_mask).mean():.0%})')

## 4. Train and evaluate models

In [ ]:
models = {
    'Linear Regression': LinearRegression(),
    'Random Forest':     RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1),
    'XGBoost':           XGBRegressor(n_estimators=200, learning_rate=0.05, max_depth=5,
                                      random_state=42, verbosity=0),
}

results = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    results[name] = {
        'R2':  r2_score(y_test, y_pred),
        'MAE': mean_absolute_error(y_test, y_pred),
        'model': model,
        'y_pred': y_pred,
    }

summary = pd.DataFrame({k: {'R²': v['R2'], 'MAE': v['MAE']} for k, v in results.items()}).T
print(summary.round(4))

## 5. Predicted vs actual

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, (name, res) in zip(axes, results.items()):
    ax.scatter(y_test, res['y_pred'], alpha=0.3, s=5)
    lims = [y_test.min(), y_test.max()]
    ax.plot(lims, lims, 'r--', linewidth=1)
    ax.set_xlabel('Actual sd_std')
    ax.set_ylabel('Predicted sd_std')
    ax.set_title(f'{name}\nR²={res["R2"]:.3f}, MAE={res["MAE"]:.5f}')

plt.tight_layout()
plt.show()

## 6. Feature importance (XGBoost)

In [ ]:
xgb = results['XGBoost']['model']
importance = pd.Series(xgb.feature_importances_, index=X_train.columns).sort_values()

fig, ax = plt.subplots(figsize=(8, 5))
importance.plot(kind='barh', ax=ax, color='steelblue')
ax.set_title('Feature importance — XGBoost')
ax.set_xlabel('Importance score')
plt.tight_layout()
plt.show()

## 7. Find optimal waveform settings

Use the best model to predict `sd_std` across all tested parameter combinations and rank by lowest predicted variability.

In [ ]:
best_model = results['XGBoost']['model']

# Build prediction grid from all unique parameter combinations in the dataset
grid = agg[condition_cols].drop_duplicates().copy()

color_dummies_grid = pd.get_dummies(grid['Color$'], prefix='color', drop_first=False)
base_grid = grid[['V', 'F_r', 'dt2', 'Coverage#']].copy()
base_grid['V_x_Fr']         = grid['V'] * grid['F_r']
base_grid['dt2_x_coverage'] = grid['dt2'] * grid['Coverage#']
base_grid['V_sq']           = grid['V'] ** 2
base_grid['Fr_sq']          = grid['F_r'] ** 2

X_grid = pd.concat([base_grid, color_dummies_grid], axis=1)

# Align columns with training data
X_grid = X_grid.reindex(columns=X_train.columns, fill_value=0)

grid['predicted_sd_std'] = best_model.predict(X_grid)

# Top 10 settings per color
top = (
    grid.groupby('Color$')
    .apply(lambda g: g.nsmallest(10, 'predicted_sd_std'))
    .reset_index(drop=True)
)
print(top[['Color$', 'V', 'F_r', 'dt2', 'Coverage#', 'predicted_sd_std']].to_string(index=False))